In [0]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, ArrayType

address_schema = StructType([
    StructField("street", StringType()),
    StructField("city", StringType()),
    StructField("state", StringType()),
    StructField("zip_code", StringType()),
    StructField("country", StringType())
])

schema = StructType([
    StructField("transaction_id", StringType()),
    StructField("user_id", StringType()),
    StructField("transaction_type", StringType()),
    StructField("timestamp", StringType()),
    StructField("status", StringType()),
    StructField("payment_method", StringType()),
    StructField("total", DoubleType()),
    StructField("currency", StringType()),
    StructField("line_items", ArrayType(StringType())),
    StructField("billing_address", address_schema),
    StructField("shipping_address", address_schema)
])

kafka_bootstrap_servers = "ec2-18-188-16-42.us-east-2.compute.amazonaws.com:9092"
topic = "transaction_events"

df = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers)
    .option("subscribe", topic)
    .option("startingOffsets", "earliest")
    .load())

json_df = df.selectExpr("CAST(value AS STRING)") \
            .select(from_json(col("value"), schema).alias("data")) \
            .select("data.*")

checkpoint_path = "/Volumes/project_2/temporary/checkpoint_transactions"
landing_zone_path = "/Volumes/project_2/datalake/landing_zone/transactions"

query = (json_df.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", checkpoint_path)
    .start(landing_zone_path))

query.awaitTermination()

display(spark.read.format("delta").load("/Volumes/project_2/datalake/landing_zone/transactions"))

In [0]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, ArrayType
 
schema = StructType([
    StructField("event_id", StringType()),
    StructField("user_id", StringType()),
    StructField("session_id", StringType()),
    StructField("event_type", StringType()),
    StructField("timestamp", StringType()),
    StructField("page", StringType()),
    StructField("device", StringType()),
    StructField("browser", StringType()),
    StructField("country", StringType()),
    StructField("city", StringType()),
    StructField("search_query", StringType()),
    StructField("element_id", StringType()),
    StructField("product_id", StringType()),
    StructField("quantity", IntegerType())
])
 
kafka_bootstrap_servers = "ec2-18-188-16-42.us-east-2.compute.amazonaws.com:9092"
topic = "user_events"
 
df = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers)
    .option("subscribe", topic)
    .option("startingOffsets", "earliest")
    .load())
 
json_df = df.selectExpr("CAST(value AS STRING)") \
            .select(from_json(col("value"), schema).alias("data")) \
            .select("data.*")
 
checkpoint_path = "/Volumes/project_2/temporary/checkpoint_users"
landing_zone_path = "/Volumes/project_2/datalake/landing_zone/users"
 
query = (json_df.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", checkpoint_path)
    .start(landing_zone_path))
 
display(spark.read.format("delta").load("/Volumes/project_2/datalake/landing_zone/users"))